# Polymarket Data Analysis for Elections

**Author**: [Johnatan Messias](https://johnnatan-messias.github.io)

**Date**: March 2026

This notebook is organized as a narrative walkthrough of the current Polymarket dataset, moving from data preparation to platform-wide market structure and then to a focused analysis of prediction markets events.


## Notebook Roadmap

The notebook now follows four sections:

1. Data preparation and schema validation.
2. Platform-wide market structure and growth.
3. Liquidity, engagement, and resolution risk.
4. Election-focused analysis.

Short markdown interpretation cells are placed after the main outputs so each chart is paired with the takeaway it is intended to support.


In [1]:
import plotly.io as pio
import polars as pl
import plotly.graph_objects as go
from IPython.display import HTML
from pathlib import Path
from plotly.subplots import make_subplots

from ast import literal_eval

pl.Config.set_engine_affinity("streaming")
pl.Config.set_fmt_str_lengths(1000)
pio.renderers.default = "notebook_connected"

In [2]:
# Setup directories for data storage
# All paths are relative to the notebook location

# Configuration
DATA_DIR = Path("../data/polymarket_dump")
PARQUET_DIR = DATA_DIR / "parquet"
PLOTS_DIR = Path("..") / "plots"

PARQUET_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Data directories initialized:")
print(f"  - Data: {DATA_DIR}")
print(f"  - Parquet: {PARQUET_DIR}")
print(f"  - Plots: {PLOTS_DIR}")

✓ Data directories initialized:
  - Data: ../data/polymarket_dump
  - Parquet: ../data/polymarket_dump/parquet
  - Plots: ../plots


In [3]:
# Add source code directory to path
import sys
SRC_DIR = Path.cwd().parent / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"✓ Source directory added to path: {SRC_DIR}")

✓ Source directory added to path: /Users/johnnatanmessias/research-projects/research-proj-prediction-markets/src


## 1. Data Preparation

The analysis starts by loading the raw event dump and expanding nested objects into separate event-, market-, series-, and tag-level tables. This matters because most later findings depend on correctly parsing timestamps, preserving identifiers, and keeping the event-to-market relationship intact.


In [4]:
events_lf = (pl
             .scan_parquet(PARQUET_DIR / "events.parquet")
             #  .filter(pl.col("closed"))
             )

markets_lf = (events_lf
              .select("markets")
              .explode("markets")
              .unnest("markets")
              .with_columns([
                  pl.col("endDate").str.to_datetime(time_zone="UTC"),
                  pl.col("startDate").str.to_datetime(time_zone="UTC"),
                  pl.col("createdAt").str.to_datetime(time_zone="UTC"),
                  pl.col("updatedAt").str.to_datetime(time_zone="UTC"),
                  pl.col("closedTime").str.to_datetime(time_zone="UTC"),
                  pl.col("endDateIso").str.to_date(),
                  pl.col("startDateIso").str.to_date(),
                  pl.col("clobTokenIds").map_elements(
                      literal_eval, return_dtype=pl.List(pl.String))
              ])
              .drop_nulls(subset="id")
              )

series_lf = (events_lf
             .select("series")
             .explode("series")
             .unnest("series")
             .with_columns([
                 pl.col("publishedAt").str.to_datetime(time_zone="UTC"),
                 pl.col("createdAt").str.to_datetime(time_zone="UTC"),
                 pl.col("updatedAt").str.to_datetime(time_zone="UTC"),
                 pl.col("startDate").str.to_datetime(time_zone="UTC"),
             ])
             .drop_nulls(subset="id")
             )

tags_lf = (events_lf
           .select("tags")
           .explode("tags")
           .unnest("tags")
           .with_columns(pl.col("updatedAt").str.to_datetime(time_zone="UTC"))
           .drop_nulls(subset="id")
           )

events_lf = (events_lf
             .with_columns([
                 pl.col("startDate").str.to_datetime(time_zone="UTC"),
                 pl.col("creationDate").str.to_datetime(time_zone="UTC"),
                 pl.col("endDate").str.to_datetime(time_zone="UTC"),
                 pl.col("createdAt").str.to_datetime(time_zone="UTC"),
                 pl.col("published_at").str.to_datetime(time_zone="UTC"),
                 pl.col("updatedAt").str.to_datetime(time_zone="UTC"),
                 pl.col("closedTime").str.to_datetime(time_zone="UTC")
             ])
             #  .drop(["markets", "series", "tags"])
             .drop_nulls(subset="id")
             )

## 2. Platform-Wide Market Structure

This section quantifies how large the platform is, how quickly activity grew, and which categories account for most of the observed market creation and trading volume.


In [5]:
def print_stats(data_stats, title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    for col in data_stats.columns:
        val = data_stats[col][0]
        if isinstance(val, (int, float)):
            if col.endswith("date"):
                print(f"{col:.<40} {val}")
            else:
                print(f"{col:.<40} {val:>15,.0f}")
        else:
            print(f"{col:.<40} {val}")

In [ ]:
events_data_stats = (events_lf
                     .select([
                         pl.len().alias("num_events"),
                         pl.col("closed").sum().alias("num_events_closed"),
                         pl.col("createdAt").min().alias(
                             "first_event_date"),
                         pl.col("createdAt").max().alias(
                             "last_event_date"),
                         pl.col("volume").sum().alias(
                             "total_event_volume"),
                         pl.col("volume").mean().alias(
                             "avg_event_volume"),
                         pl.col("electionType").n_unique().alias(
                             "num_election_type"),
                         pl.col("commentCount").sum().alias(
                             "total_event_comments"),
                         pl.col("commentCount").mean().alias(
                             "avg_comments_per_event"),
                         pl.col("markets").list.len().sum().alias(
                             "num_markets_with_events"),
                         pl.col("series").list.len().sum().alias("num_series"),
                         pl.col("series").list.len().mean().alias(
                             "avg_series_per_event"),
                         pl.col("tags").list.len().sum().alias("num_tags"),
                         pl.col("tags").list.len().mean().alias(
                             "avg_tags_per_event"),
                     ])
                     .collect()
                     )

market_data_stats = (markets_lf
                     .select([
                         pl.len().alias("num_markets"),
                         pl.col("createdAt").min().alias(
                             "first_market_date"),
                         pl.col("createdAt").max().alias(
                             "last_market_date"),
                         pl.col("volumeNum").sum().alias("total_volume"),
                         pl.col("volumeNum").mean().alias(
                             "avg_volume_per_market"),
                         pl.col("creator").n_unique().alias("num_creators"),
                         pl.col("creator").unique().sort().implode().list.join(", ").alias(
                             "creators_list"),
                         pl.col("category").n_unique().alias("num_categories"),
                         pl.col("category").unique().sort().implode().list.join(", ").alias(
                             "category_list"),
                         pl.col("marketType").n_unique().alias(
                             "num_market_types"),
                         pl.col("marketType").unique().sort().implode().list.join(", ").alias(
                             "types_list"),
                         pl.col("clobTokenIds").list.len().min().alias(
                             "min_num_clob_token_ids"),
                         pl.col("clobTokenIds").list.len().max().alias(
                             "max_num_clob_token_ids"),
                         pl.col("clobTokenIds").list.len().mean().alias(
                             "avg_num_clob_token_ids"),
                         pl.col("clobTokenIds").list.explode().unique().len().alias(
                             "total_num_clob_token_ids"),
                     ])
                     .collect()
                     )

series_data_stats = (series_lf
                     .select([
                         pl.len().alias("num_series"),
                         pl.col("createdAt").min().alias(
                             "first_series_date"),
                         pl.col("createdAt").max().alias(
                             "last_series_date"),
                         pl.col("title").n_unique().alias(
                             "num_unique_series_titles"),
                         pl.col("commentCount").sum().alias(
                             "total_series_comments"),
                         pl.col("commentCount").mean().alias(
                             "avg_comments_per_series"),
                         pl.col("seriesType").n_unique().alias(
                             "num_series_types"),
                         pl.col("seriesType").unique().sort().implode().list.join(", ").alias(
                             "series_types_list"),
                         pl.col("recurrence").n_unique().alias(
                             "num_recurrence_types"),
                         pl.col("recurrence").unique().sort().implode().list.join(", ").alias(
                             "recurrence_types_list"),
                     ])
                     .collect()
                     )

tags_data_stats = (tags_lf
                   .select([
                       pl.len().alias("num_tags"),
                       pl.col("updatedAt").min().alias(
                           "first_tag_date"),
                       pl.col("updatedAt").max().alias(
                           "last_tag_date"),
                       pl.col("label").n_unique().alias(
                           "num_unique_tag_labels")
                   ])
                   .collect()
                   )

print_stats(events_data_stats, title="EVENTS STATISTICS")
print_stats(market_data_stats, title="MARKETS STATISTICS")
print_stats(series_data_stats, title="SERIES STATISTICS")
print_stats(tags_data_stats, title="TAGS STATISTICS")